In [127]:
import pandas as pd
import numpy as np
from combat.pycombat import pycombat  # pip install combat
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.express as px

In [128]:
# To make plotly fig show in notebook
import plotly.io as pio
pio.renderers.default = "notebook"

# MAIN

In [129]:
# Read filtered TSV (only non fully '0' rows)
X = pd.read_csv("CUSTOM_hg38_episign/meth_matrix.tsv", sep="\t", index_col=0)

# Compute epiSize (= size of epiSign):
epiSize = {}
for epiSign in [x for x in X.columns if x not in ('coord')]:
    epiSize[epiSign] = sum(X[epiSign] > 99)  # Catch 100% methyl

## Pre-processing

## Batch-effect correction using (py)combat

In [156]:
# First we generate the list of batches:
ref_sign = ["ADCADN","ATRX","AUTS18","BAFopathy","BFLS","CHARGE","CdLS","Down","Dup7","EEOC","FLHS","GTPTS","HMA","HVDAS_C","HVDAS_T","ICF1","ICF2_3_4","KDVS","Kabuki","Kleefstra","MRD51","MRX93","MRX97","MRXCJS","MRXSN","MRXSSR","RMNS","RSTS","SBBYSS","SETD1B","Sotos","TBRS","WDSTS","Williams"]
dataset_251217 = ["HG002_combined","barcode04_combined"]

batch = []
datasets = [ref_sign, dataset_251217]
for j in range(len(datasets)):
    batch.extend([j for _ in range(len(datasets[j]))])

# Then run (py)combat:
X_corrected = pycombat(X, batch)

Found 2 batches.
Adjusting for 0 covariate(s) or covariate level(s).
Standardizing Data across genes.
Fitting L/S model and finding priors.
Finding parametric adjustments.
Adjusting the Data


/home/felix/.local/share/mamba/envs/NSBEpi/lib/python3.9/site-packages/combat/pycombat.py:159: RuntimeWarning: divide by zero encountered in divide
  np.absolute(d_new-d_old)/d_old))  # maximum difference between new and old estimate


## Transpose + n

In [147]:
# Transpose
X_t = X_corrected.T  # Required
print(X_t.index)

to_PCA = X_t
if 'epiSize' in X_t.columns:
    to_PCA = X_t.drop('epiSize', axis=1)

# Remove 2nd row = size of epiSignormalize) then normalize
X_scaled = StandardScaler().fit_transform(to_PCA)

Index(['ADCADN', 'ATRX', 'AUTS18', 'BAFopathy', 'BFLS', 'CHARGE', 'CdLS',
       'Down', 'Dup7', 'EEOC', 'FLHS', 'GTPTS', 'HMA', 'HVDAS_C', 'HVDAS_T',
       'ICF1', 'ICF2_3_4', 'KDVS', 'Kabuki', 'Kleefstra', 'MRD51', 'MRX93',
       'MRX97', 'MRXCJS', 'MRXSN', 'MRXSSR', 'RMNS', 'RSTS', 'SBBYSS',
       'SETD1B', 'Sotos', 'TBRS', 'WDSTS', 'Williams', 'HG002_combined',
       'barcode04_combined'],
      dtype='object')


## PCA

In [148]:
# Run PCA:
NB_COMPON = 3
pca = PCA(n_components=NB_COMPON)
pcs = pca.fit_transform(to_PCA)

# Make a dict with '% variance explained' for each component:
dict_compon = {'compon'+str(i) : str(round(pca.explained_variance_ratio_[i]*100,4)) for i in range(NB_COMPON)}

In [149]:
# Top N features of each componennt
compon_0_top = np.abs(pca.components_[0]).argsort()[::-1][:5]
print("Component 0:", list(X.index[compon_0_top]))

compon_1_top = np.abs(pca.components_[1]).argsort()[::-1][:5]
print("Component 1:", list(X.index[compon_1_top]))

Component 0: ['3:155853019-155853020', '14:57391664-57391665', '13:111154601-111154602', '3:57757435-57757436', '18:50286803-50286804']
Component 1: ['13:110044472-110044473', '15:51234222-51234223', '13:26840082-26840083', '7:27186709-27186710', '7:27090704-27090705']


In [150]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
pcs_df = pd.DataFrame(
    pcs,
    index=X.columns,
    columns=dict_compon.keys()
)
print(pcs_df.loc[['Kabuki', 'barcode04_combined', 'HG002_combined']])

                       compon0     compon1     compon2
Kabuki              932.116091  433.916609 -471.616066
barcode04_combined  -19.092736   -0.615051   -6.042973
HG002_combined      -12.413559  -35.096866  -10.195378


In [151]:
# Plot PCA
color_selected = [ x in ['RMNS','Kleefstra','Kabuki','barcode04_combined','HG002_combined'] for x in pcs_df.index ]

x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df,
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=color_selected,
        labels={x_compon:':'.join([x_compon,dict_compon[x_compon]]), y_compon:':'.join([y_compon,dict_compon[y_compon]])}
)
fig.show()

## t-SNE

In [152]:
# Run t-SNE:
# MEMOs:
# - In Joris' paper they use 'preplex=2'
# - t-SNE is stochastic -> re-run multiple times ?
#
tsne = TSNE(n_components=2, perplexity=30).fit_transform(X_scaled)

In [153]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
tsne_df = pd.DataFrame(
    tsne,
    index=X.columns,
    columns=('compon0', 'compon1')
)
subset_tsne = tsne_df.loc[['Kabuki', 'barcode04_combined', 'HG002_combined']]
print(subset_tsne)

                      compon0    compon1
Kabuki             -15.311996  21.950497
barcode04_combined   3.940955  -3.526023
HG002_combined      -1.136360   0.608769


In [154]:
# Plot t-SNE
color_selected = [x in ['RMNS','Kleefstra', 'Kabuki', 'barcode04_combined', 'HG002_combined'] for x in tsne_df.index]
fig = px.scatter(
        tsne_df,
        x='compon0',
        y='compon1',
        hover_data=[tsne_df.index],
        color=color_selected
)
fig.show()